# NephroLLM — Execution Plan

1. Environment setup
2. Load Llama 3.2 3B Instruct
3. Verify base-model inference
4. Load and validate the nephrology benchmark
5. Run a fresh baseline evaluation
6. Analyze baseline failures
7. Prepare fine-tuning dataset
8. Fine-tune using QLoRA
9. Save the LoRA adapter
10. Run the exact same evaluation on the fine-tuned model
11. Evaluate on held-out questions
12. Compare base vs fine-tuned model
13. Document findings, limitations, and conclusions

# NephroLLM — Fine-Tuning Llama 3.2 3B for Nephrology Q&A

Educational fine-tuning experiment using QLoRA to adapt Llama 3.2 3B Instruct to nephrology-focused questions.

**Objective:** Compare the base model against the QLoRA fine-tuned model using the same evaluation benchmark and evaluation protocol.

In [37]:
# cell 2
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl

cell 3
**package/GPU/version check**

In [38]:
# cell 4
import torch
import transformers
import datasets
import peft
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.10.0+cu128
Transformers: 5.17.0
Datasets: 5.0.1
PEFT: 0.21.0
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


cell 5 **Hugging Face authentication**

In [39]:
# cell 6
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

print("HF token available:", hf_token is not None)
print("Token length:", len(hf_token) if hf_token else 0)

HF token available: True
Token length: 37


cell 7 # **Step 2 — Load Llama 3.2 3B**

In [40]:
# cell 8 
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient

# Get Hugging Face token securely
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print("Model loaded successfully!")
print("Model device:", model.device)

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded successfully!
Model device: cuda:0


cell 9 

**Step 3 — Run our first baseline inference**

Now we're going to talk to the untouched model.

This is important: don't fine-tune anything yet. We first want to see what the base model does.


                    BASE MODEL
                       │
                       ▼
             30 nephrology questions
                       │
                       ▼
                Baseline scores
                       │
                       ▼
                Failure analysis
                       │
                       ▼
                  QLoRA training
                       │
                       ▼
              SAME 30 questions
                       │
                       ▼
             Fine-tuned scores

cell 10 
**ask_model()**

In [41]:
# cell 11
def ask_model(question, max_new_tokens=256):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an educational assistant specializing in nephrology. "
                "Provide clear, factual, educational explanations. "
                "Do not provide diagnosis or personalized medical advice."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

cell 12 **smoke test**

In [43]:
# cell 13
question = "What is the primary function of the glomerulus?"

answer = ask_model(question)

print("QUESTION:")
print(question)

print("\nMODEL ANSWER:")
print(answer)

QUESTION:
What is the primary function of the glomerulus?

MODEL ANSWER:
The primary function of the glomerulus is to filter waste and excess fluids from the blood. It is a critical component of the nephron, the functional unit of the kidney, and plays a vital role in maintaining overall kidney function.

The glomerulus is a tuft of capillaries that receives blood from the afferent arteriole and sends it to the efferent arteriole. As the blood flows through the glomerulus, it is subjected to a pressure gradient that forces the fluid to pass through the glomerular basement membrane and into the Bowman's capsule. This process is known as filtration.

The glomerulus filters out waste products, excess ions, and excess water from the blood, while retaining essential nutrients, proteins, and other vital substances. The filtered waste products, including urea, creatinine, and other waste products, are then transported to the renal tubules for further processing and excretion.

In essence, the

cell 14 
**NephroLLM data strategy**

| Purpose                  | Data                                               | Use                         |
| ------------------------ | -------------------------------------------------- | --------------------------- |
| **Primary benchmark**    | NEJM-AI nephrology dataset                         | Baseline + final evaluation |
| **Fine-tuning**          | Our curated nephrology educational Q&A             | Training                    |
| **Generalization check** | Separate nephrology questions not used in training | Held-out evaluation         |



                 NEPHROLOGY BENCHMARK
                         │
          ┌──────────────┼──────────────┐
          │              │              │
          ▼              ▼              ▼
      Baseline/       Held-out       Training
      Final Test      Evaluation      Dataset
          │              │              │
          │              │              └── QLoRA
          │              │
          ▼              ▼
      Base vs FT      Generalization

cell 15 # Step 4 — Run the Llama 3.2 3B baseline on the 636 questions

In [44]:
# cell 16
import os
import pandas as pd

# Find the finalized primary evaluation file
matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "nephrollm_primary_eval.csv":
            matches.append(os.path.join(root, file))

print("Found:", matches)

if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one primary evaluation file, found: {matches}"
    )

eval_path = matches[0]
eval_df = pd.read_csv(eval_path)

print("Evaluation file:", eval_path)
print("Questions loaded:", len(eval_df))

display(
    eval_df[
        ["BenchmarkID", "Subject", "Question", "Choices", "Answer"]
    ].head()
)

Found: ['/kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark/nephrollm_primary_eval.csv']
Evaluation file: /kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark/nephrollm_primary_eval.csv
Questions loaded: 636


,BenchmarkID,Subject,Question,Choices,Answer
0,NEPHRO-EVAL-0001,Acute Kidney Injury & Critical Care Nephrology,Question 2824:\nThe transfusion of longer-term...,1. Increased risk of transfusion-related react...,B
1,NEPHRO-EVAL-0002,Acute Kidney Injury & Critical Care Nephrology,"Based on data from the NUTRIREA-2 study, enter...",A. Decreased incidence of systemic bacteremia\...,B
2,NEPHRO-EVAL-0003,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is CORRECT about th...,1. She is at decreased risk for inpatient mort...,B
3,NEPHRO-EVAL-0004,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is associated with ...,A. A decreased risk of surgical site infection...,B
4,NEPHRO-EVAL-0005,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is the MOST importa...,A. Administration of a bolus of 30 ml/kg 0.9% ...,D


cell 17
**Step 5 — Run the full 636-question baseline**

Before doing that, let's lock the evaluation configuration.

We'll use:

636 primary questions
Same prompt for every question
do_sample=False → deterministic generation
max_new_tokens=96
Raw response saved
Extracted answer saved
Ground truth saved
Accuracy calculated automatically
Results saved to CSV**
Before doing that, let's lock the evaluation configuration.

We'll use:

636 primary questions
Same prompt for every question
do_sample=False → deterministic generation
max_new_tokens=96
Raw response saved
Extracted answer saved
Ground truth saved
Accuracy calculated automatically
Results saved to CSV

cell 18 ## Baseline MCQ Evaluator

The benchmark contains multiple-choice nephrology questions.
Questions may contain four or five answer choices, and the source
dataset may label them either alphabetically or numerically.

The evaluator converts the available choices into a consistent
A/B/C/D/E representation and requires the model to return only
the selected option.

In [63]:
import re

def extract_answer(response):
    """
    Extract a single answer choice.

    Supports alphabetical and numerical answer formats.
    Numerical choices are mapped by position:
    1=A, 2=B, 3=C, 4=D, 5=E
    """

    if not response:
        return None

    text = str(response).strip().upper()

    # 1. Explicit ANSWER / CORRECT ANSWER formats
    patterns = [
        r"\bANSWER\s*:\s*([A-E])\b",
        r"\bCORRECT ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
        r"\bTHE ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
        r"\bMY ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)

    # 2. Explicit numeric ANSWER format
    numeric_patterns = [
        r"\bANSWER\s*:\s*([1-5])\b",
        r"\bCORRECT ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
        r"\bTHE ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
        r"\bMY ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
    ]

    for pattern in numeric_patterns:
        match = re.search(pattern, text)
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    # 3. Standalone answer formats: A, A., A)
    lines = [x.strip() for x in text.splitlines() if x.strip()]

    for line in lines[:8]:
        match = re.fullmatch(r"([A-E])[\.\)]?", line)
        if match:
            return match.group(1)

    # 4. Standalone numeric answer: 1, 1., 1)
    for line in lines[:8]:
        match = re.fullmatch(r"([1-5])[\.\)]?", line)
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    # 5. OPTION / CHOICE format
    for line in lines[:8]:
        match = re.fullmatch(
            r"(?:OPTION|CHOICE)\s*[:\-]?\s*([A-E])[\.\)]?",
            line
        )
        if match:
            return match.group(1)

        match = re.fullmatch(
            r"(?:OPTION|CHOICE)\s*[:\-]?\s*([1-5])[\.\)]?",
            line
        )
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    return None

In [57]:
# cell 19
def ask_mcq_fast(question, choices, max_new_tokens=32):
    """
    Ask the base model to answer one MCQ.

    The benchmark may contain 4 or 5 choices and may use
    alphabetical or numerical labels.
    """

    choice_lines = str(choices).strip()

    prompt = f"""Answer this multiple-choice nephrology question.

Question:
{question}

Choices:
{choice_lines}

Select the single best answer from the choices provided.

Return ONLY:
ANSWER: <option letter>
"""

    return ask_model(
        prompt,
        max_new_tokens=max_new_tokens
    )

In [58]:
# cell 27 
test_df = eval_df.head(5).copy()

test_results = []

for _, row in test_df.iterrows():

    response = ask_mcq_fast(
        row["Question"],
        row["Choices"]
    )

    predicted = extract_answer(response)
    expected = str(row["Answer"]).strip().upper()

    test_results.append({
        "BenchmarkID": row["BenchmarkID"],
        "Expected": expected,
        "Predicted": predicted,
        "Correct": predicted == expected,
        "ModelResponse": response
    })

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

,BenchmarkID,Expected,Predicted,Correct,ModelResponse
0,NEPHRO-EVAL-0001,B,D,False,D
1,NEPHRO-EVAL-0002,B,A,False,A
2,NEPHRO-EVAL-0003,B,A,False,A.
3,NEPHRO-EVAL-0004,B,B,True,B
4,NEPHRO-EVAL-0005,D,D,True,D


cell 20 ### Evaluator Smoke Test

Run a small sample before the full benchmark.

In [59]:
from tqdm.auto import tqdm
import os
import pandas as pd

BATCH_SIZE = 50

BASELINE_PREFIX = "/kaggle/working/nephrollm_baseline_final_batch_"

for start in range(0, len(eval_df), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(eval_df))
    batch = eval_df.iloc[start:end].copy()

    batch_number = start // BATCH_SIZE + 1

    print(
        f"\nRunning Batch {batch_number}: "
        f"questions {start + 1} to {end}"
    )

    batch_results = []

    for _, row in tqdm(
        batch.iterrows(),
        total=len(batch),
        desc=f"Batch {batch_number}"
    ):

        response = ask_mcq_fast(
            row["Question"],
            row["Choices"]
        )

        predicted = extract_answer(response)
        expected = str(row["Answer"]).strip().upper()

        batch_results.append({
            "BenchmarkID": row["BenchmarkID"],
            "Subject": row["Subject"],
            "Question": row["Question"],
            "Choices": row["Choices"],
            "Expected": expected,
            "Predicted": predicted,
            "Correct": predicted == expected,
            "ModelResponse": response
        })

    batch_df = pd.DataFrame(batch_results)

    batch_path = (
        f"{BASELINE_PREFIX}"
        f"{start + 1:03d}_{end:03d}.csv"
    )

    batch_df.to_csv(batch_path, index=False)

    print(f"Saved: {batch_path}")
    print(
        f"Batch accuracy: "
        f"{batch_df['Correct'].mean():.2%}"
    )
    print(
        f"Unparsed: "
        f"{batch_df['Predicted'].isna().sum()}"
    )


Running Batch 1: questions 1 to 50


Batch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_001_050.csv
Batch accuracy: 40.00%
Unparsed: 19

Running Batch 2: questions 51 to 100


Batch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_051_100.csv
Batch accuracy: 48.00%
Unparsed: 1

Running Batch 3: questions 101 to 150


...

Saved: /kaggle/working/nephrollm_baseline_final_batch_501_550.csv
Batch accuracy: 14.00%
Unparsed: 20

Running Batch 12: questions 551 to 600


Batch 12:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_551_600.csv
Batch accuracy: 24.00%
Unparsed: 23

Running Batch 13: questions 601 to 636


Batch 13:   0%|          | 0/36 [00:00<?, ?it/s]

Saved: /kaggle/working/nephrollm_baseline_final_batch_601_636.csv
Batch accuracy: 11.11%
Unparsed: 29


In [60]:
import glob
import pandas as pd

baseline_files = sorted(
    glob.glob(
        "/kaggle/working/nephrollm_baseline_final_batch_*.csv"
    )
)

print("Fresh baseline batch files:", len(baseline_files))

if len(baseline_files) != 13:
    raise ValueError(
        f"Expected 13 fresh baseline files, found {len(baseline_files)}"
    )

baseline_df = pd.concat(
    [pd.read_csv(f) for f in baseline_files],
    ignore_index=True
)

# Safety checks
print("Rows before deduplication:", len(baseline_df))
print(
    "Unique BenchmarkIDs:",
    baseline_df["BenchmarkID"].nunique()
)

if len(baseline_df) != 636:
    raise ValueError(
        f"Expected 636 rows, found {len(baseline_df)}"
    )

if baseline_df["BenchmarkID"].nunique() != 636:
    raise ValueError("Duplicate or missing BenchmarkIDs detected.")

# Save one consolidated fresh baseline file
baseline_path = (
    "/kaggle/working/"
    "nephrollm_baseline_final_results.csv"
)

baseline_df.to_csv(
    baseline_path,
    index=False
)

print("\nFresh baseline saved:")
print(baseline_path)

print("\nColumns:")
print(list(baseline_df.columns))

Fresh baseline batch files: 13
Rows before deduplication: 636
Unique BenchmarkIDs: 636

Fresh baseline saved:
/kaggle/working/nephrollm_baseline_final_results.csv

Columns:
['BenchmarkID', 'Subject', 'Question', 'Choices', 'Expected', 'Predicted', 'Correct', 'ModelResponse']


In [61]:
print("NEPHROLLM — FRESH BASELINE")
print("==========================")

total_questions = len(baseline_df)
correct = baseline_df["Correct"].sum()
unparsed = baseline_df["Predicted"].isna().sum()
incorrect = total_questions - correct - unparsed

accuracy_all = correct / total_questions
accuracy_answered = (
    correct / (total_questions - unparsed)
    if total_questions - unparsed > 0
    else 0
)

print(f"Total questions:      {total_questions}")
print(f"Correct:              {correct}")
print(f"Incorrect:            {incorrect}")
print(f"Unparsed:             {unparsed}")
print(f"Accuracy (all):       {accuracy_all:.2%}")
print(f"Accuracy (answered):  {accuracy_answered:.2%}")

NEPHROLLM — FRESH BASELINE
Total questions:      636
Correct:              183
Incorrect:            232
Unparsed:             221
Accuracy (all):       28.77%
Accuracy (answered):  44.10%


In [62]:
unparsed_df = baseline_df[
    baseline_df["Predicted"].isna()
].copy()

print("Unparsed responses:", len(unparsed_df))

display(
    unparsed_df[
        [
            "BenchmarkID",
            "Expected",
            "Choices",
            "ModelResponse"
        ]
    ].head(20)
)

Unparsed responses: 221


,BenchmarkID,Expected,Choices,ModelResponse
24,NEPHRO-EVAL-0025,C,A. Intermittent hemodiaﬁltration\nB. Intermitt...,I'm happy to help you with your nephrology que...
27,NEPHRO-EVAL-0028,B,1. Lower serum ammonia level\n2. Improved ment...,ANSWER: 3
28,NEPHRO-EVAL-0029,B,1. Norepinephrine\n2. Midodrine plus octreotid...,ANSWER: 2
29,NEPHRO-EVAL-0030,D,1. Electronic alerts that identify patients wh...,ANSWER: 3
30,NEPHRO-EVAL-0031,D,1. Regional citrate anticoagulation\n2. Unfrac...,ANSWER: 1
32,NEPHRO-EVAL-0033,D,1. Improved survival\n2. Decreased risk of AKI...,ANSWER: 3
33,NEPHRO-EVAL-0034,B,1. 0.9% Saline\n2. Isotonic balanced electroly...,ANSWER: 4
34,NEPHRO-EVAL-0035,C,1. Increased risk of AKI\n2. Decreased number ...,ANSWER: 3
35,NEPHRO-EVAL-0036,A,1. Decreased rate of progression to stage 2 AK...,ANSWER: 1
36,NEPHRO-EVAL-0037,B,1. Her age\n2. Her bilirubinuria\n3. Her criti...,ANSWER: 1


In [64]:
baseline_df["Predicted"] = (
    baseline_df["ModelResponse"]
    .apply(extract_answer)
)

baseline_df["Correct"] = (
    baseline_df["Predicted"]
    == baseline_df["Expected"]
)

total_questions = len(baseline_df)
correct = baseline_df["Correct"].sum()
unparsed = baseline_df["Predicted"].isna().sum()
incorrect = total_questions - correct - unparsed

print("NEPHROLLM — FRESH BASELINE (REPARSED)")
print("=====================================")
print(f"Total questions:      {total_questions}")
print(f"Correct:              {correct}")
print(f"Incorrect:            {incorrect}")
print(f"Unparsed:             {unparsed}")
print(f"Accuracy (all):       {correct / total_questions:.2%}")
print(
    f"Accuracy (answered):  "
    f"{correct / (total_questions - unparsed):.2%}"
)

baseline_df.to_csv(
    "/kaggle/working/nephrollm_baseline_final_results.csv",
    index=False
)

NEPHROLLM — FRESH BASELINE (REPARSED)
Total questions:      636
Correct:              264
Incorrect:            328
Unparsed:             44
Accuracy (all):       41.51%
Accuracy (answered):  44.59%


In [65]:
remaining_unparsed = baseline_df[
    baseline_df["Predicted"].isna()
].copy()

print("Remaining unparsed:", len(remaining_unparsed))

display(
    remaining_unparsed[
        [
            "BenchmarkID",
            "Expected",
            "Choices",
            "ModelResponse"
        ]
    ]
)

Remaining unparsed: 44


,BenchmarkID,Expected,Choices,ModelResponse
24,NEPHRO-EVAL-0025,C,A. Intermittent hemodiaﬁltration\nB. Intermitt...,I'm happy to help you with your nephrology que...
64,NEPHRO-EVAL-0065,A,A. Regional citrate anticoagulation (RC...,I'm happy to help you with your nephrology que...
144,NEPHRO-EVAL-0145,D,A. Denosumab\nB. Increase calcium acetate\nC. ...,I'm happy to help you with your nephrology que...
155,NEPHRO-EVAL-0156,A,A. Cinacalcet therapy\nB. Calcitonin\nC. Estro...,I'm happy to help you with your nephrology que...
176,NEPHRO-EVAL-0177,C,1. Refer for cardiac catheterization\n2. Chest...,I'm happy to help you with your nephrology que...
180,NEPHRO-EVAL-0181,C,1. Ankylosing spondylitis\n2. Adynamic bone di...,I'm happy to help you with your nephrology que...
182,NEPHRO-EVAL-0183,D,1. Hypercalciuria\n2. Renal tubular acidosis\n...,I'm happy to help you with your nephrology que...
188,NEPHRO-EVAL-0189,C,1. A loss of function mutation in the calcium-...,I'm happy to help you with your nephrology que...
191,NEPHRO-EVAL-0192,A,1. Change calcium acetate to sevelamer carbona...,I can't provide a specific answer without know...
227,NEPHRO-EVAL-0228,D,A. Discontinue pravastatin now\nB. Discontinue...,I'm happy to help you with your nephrology que...


In [66]:
subject_accuracy = (
    baseline_df
    .groupby("Subject")
    .agg(
        Questions=("BenchmarkID", "count"),
        Correct=("Correct", "sum"),
        Unparsed=("Predicted", lambda x: x.isna().sum())
    )
    .reset_index()
)

subject_accuracy["Incorrect"] = (
    subject_accuracy["Questions"]
    - subject_accuracy["Correct"]
    - subject_accuracy["Unparsed"]
)

subject_accuracy["Accuracy"] = (
    subject_accuracy["Correct"]
    / subject_accuracy["Questions"]
    * 100
)

subject_accuracy = subject_accuracy.sort_values(
    "Accuracy",
    ascending=True
)

display(subject_accuracy)


,Subject,Questions,Correct,Unparsed,Incorrect,Accuracy
15,Primary & Secondary Glomerular Diseases,23,6,7,10,26.086957
17,Primary and Secondary Glomerular Diseases,48,13,5,30,27.083333
4,Disorders of Divalent Ions Renal Bone Disease ...,22,6,2,14,27.272727
14,Pregnancy and Kidney Disease,24,7,3,14,29.166667
16,Primary Glomerular Disease,24,7,2,15,29.166667
7,Electrolyte Acid-Base Disorders,24,7,5,12,29.166667
5,"Disorders of Divalent Ions, Renal Bone Disease...",46,14,5,27,30.434783
8,Electrolytes and Acid-Base Disorders,22,7,2,13,31.818182
12,Infection Control & Prevention in Outpatient H...,24,9,2,13,37.500000
19,Transplantation,48,19,1,28,39.583333


In [67]:
# Baseline error analysis — parseable incorrect answers only

baseline_errors = baseline_df[
    (baseline_df["Predicted"].notna()) &
    (baseline_df["Correct"] == False)
].copy()

print("BASELINE ERROR ANALYSIS")
print("=======================")
print(f"Total benchmark questions: {len(baseline_df)}")
print(f"Parseable incorrect answers: {len(baseline_errors)}")
print()

print("Most common predicted answers:")
display(
    baseline_errors["Predicted"]
    .value_counts()
    .rename_axis("Predicted")
    .reset_index(name="Count")
)

print("\nExpected answer distribution:")
display(
    baseline_errors["Expected"]
    .value_counts()
    .rename_axis("Expected")
    .reset_index(name="Count")
)

print("\nSample baseline errors:")
display(
    baseline_errors[
        [
            "BenchmarkID",
            "Subject",
            "Question",
            "Expected",
            "Predicted",
            "ModelResponse"
        ]
    ].head(20)
)

BASELINE ERROR ANALYSIS
Total benchmark questions: 636
Parseable incorrect answers: 328

Most common predicted answers:


,Predicted,Count
0,B,123
1,A,84
2,C,55
3,D,53
4,E,13



Expected answer distribution:


,Expected,Count
0,D,95
1,C,89
2,B,73
3,A,57
4,E,14



Sample baseline errors:


,BenchmarkID,Subject,Question,Expected,Predicted,ModelResponse
0,NEPHRO-EVAL-0001,Acute Kidney Injury & Critical Care Nephrology,Question 2824:\nThe transfusion of longer-term...,B,D,D
1,NEPHRO-EVAL-0002,Acute Kidney Injury & Critical Care Nephrology,"Based on data from the NUTRIREA-2 study, enter...",B,A,A
2,NEPHRO-EVAL-0003,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is CORRECT about th...,B,A,A.
5,NEPHRO-EVAL-0006,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following individuals is MOST...,B,C,ANSWER: C
6,NEPHRO-EVAL-0007,Acute Kidney Injury & Critical Care Nephrology,"Based on the existing literature, implementing...",B,D,ANSWER: D
7,NEPHRO-EVAL-0008,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following outcomes has been d...,C,B,B
8,NEPHRO-EVAL-0009,Acute Kidney Injury & Critical Care Nephrology,Question 2833:\nImplementing a Kidney Disease ...,B,D,ANSWER: D
16,NEPHRO-EVAL-0017,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following should you tell the...,B,A,ANSWER: A
27,NEPHRO-EVAL-0028,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following is MOST likely to r...,B,C,ANSWER: 3
29,NEPHRO-EVAL-0030,Acute Kidney Injury & Critical Care Nephrology,Which ONE of the following interventions would...,D,C,ANSWER: 3


In [68]:
# Analyze baseline response formats

def classify_response_format(response):
    if response is None or str(response).strip() == "":
        return "empty"

    text = str(response).strip().upper()

    if re.search(r"\bANSWER\s*:\s*[A-E]\b", text):
        return "ANSWER: letter"

    if re.search(r"\bANSWER\s*:\s*[1-5]\b", text):
        return "ANSWER: number"

    if re.fullmatch(r"[A-E][\.\)]?", text):
        return "standalone letter"

    if re.fullmatch(r"[1-5][\.\)]?", text):
        return "standalone number"

    if text.startswith("I CANNOT") or text.startswith("I CAN'T"):
        return "refusal"

    if "MEDICAL ADVICE" in text:
        return "medical-advice refusal"

    return "other"


baseline_df["ResponseFormat"] = (
    baseline_df["ModelResponse"]
    .apply(classify_response_format)
)

format_summary = (
    baseline_df
    .groupby("ResponseFormat")
    .agg(
        Questions=("BenchmarkID", "count"),
        Correct=("Correct", "sum")
    )
    .reset_index()
)

format_summary["Accuracy"] = (
    format_summary["Correct"]
    / format_summary["Questions"]
    * 100
)

format_summary = format_summary.sort_values(
    "Questions",
    ascending=False
)

display(format_summary)

,ResponseFormat,Questions,Correct,Accuracy
0,ANSWER: letter,233,94,40.343348
1,ANSWER: number,176,81,46.022727
4,standalone letter,172,85,49.418605
2,other,46,3,6.521739
3,refusal,8,0,0.000000
5,standalone number,1,1,100.000000


In [69]:
import os
import glob

print("Kaggle input datasets:")
print("======================")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:20]:
        print(f"{indent}  {file}")

print("\nWorking directory CSV files:")
print("============================")

for path in sorted(glob.glob("/kaggle/working/*.csv")):
    print(path)

Kaggle input datasets:
input/
  datasets/
    shantanuss/
      nejm-ai-nephrology-benchmark/
        nephrollm_evaluation_manifest.csv
        nephrollm_primary_eval.csv
        nephrollm_heldout_eval.csv

Working directory CSV files:
/kaggle/working/nephrollm_baseline_final_batch_001_050.csv
/kaggle/working/nephrollm_baseline_final_batch_051_100.csv
/kaggle/working/nephrollm_baseline_final_batch_101_150.csv
/kaggle/working/nephrollm_baseline_final_batch_151_200.csv
/kaggle/working/nephrollm_baseline_final_batch_201_250.csv
/kaggle/working/nephrollm_baseline_final_batch_251_300.csv
/kaggle/working/nephrollm_baseline_final_batch_301_350.csv
/kaggle/working/nephrollm_baseline_final_batch_351_400.csv
/kaggle/working/nephrollm_baseline_final_batch_401_450.csv
/kaggle/working/nephrollm_baseline_final_batch_451_500.csv
/kaggle/working/nephrollm_baseline_final_batch_501_550.csv
/kaggle/working/nephrollm_baseline_final_batch_551_600.csv
/kaggle/working/nephrollm_baseline_final_batch_601_636.c

In [70]:
import pandas as pd

BASE = "/kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark"

manifest_path = f"{BASE}/nephrollm_evaluation_manifest.csv"
heldout_path = f"{BASE}/nephrollm_heldout_eval.csv"

manifest_df = pd.read_csv(manifest_path)
heldout_df = pd.read_csv(heldout_path)

print("MANIFEST")
print("========")
print("Shape:", manifest_df.shape)
print("Columns:", list(manifest_df.columns))
display(manifest_df.head(20))

print("\nHELD-OUT EVALUATION")
print("===================")
print("Shape:", heldout_df.shape)
print("Columns:", list(heldout_df.columns))
display(heldout_df.head(10))

MANIFEST
Shape: (796, 10)
Columns: ['BenchmarkID', 'ID', 'Subject', 'Context', 'Question', 'Choices', 'Answer', 'Solution', 'explanation', 'split']


,BenchmarkID,ID,Subject,Context,Question,Choices,Answer,Solution,explanation,split
0,NEPHRO-EVAL-0001,2824,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2824:\nThe transfusion of longer-term...,1. Increased risk of transfusion-related react...,B,temp nan,Answer B: No difference in mortality at 90 days,primary_eval
1,NEPHRO-EVAL-0002,2825,Acute Kidney Injury & Critical Care Nephrology,A 79-year-old man with stage G4 CKD and recent...,"Based on data from the NUTRIREA-2 study, enter...",A. Decreased incidence of systemic bacteremia\...,B,temp nan,Answer B: Increased incidence of gastrointest...,primary_eval
2,NEPHRO-EVAL-0003,2826,Acute Kidney Injury & Critical Care Nephrology,A 67-year-old woman with a history of stage G2...,Which ONE of the following is CORRECT about th...,1. She is at decreased risk for inpatient mort...,B,temp nan,Answer B: Her duration of RRT will be shorter,primary_eval
3,NEPHRO-EVAL-0004,2827,Acute Kidney Injury & Critical Care Nephrology,You are caring for a 71-year-old man with stag...,Which ONE of the following is associated with ...,A. A decreased risk of surgical site infection...,B,temp nan,Answer B: An increased risk of AKI requiring RRT,primary_eval
4,NEPHRO-EVAL-0005,2829,Acute Kidney Injury & Critical Care Nephrology,You are admitting a 65-year-old kidney trans-p...,Which ONE of the following is the MOST importa...,A. Administration of a bolus of 30 ml/kg 0.9% ...,D,temp nan,Answer D: Early adminstration of appropriate ...,primary_eval
5,NEPHRO-EVAL-0006,2830,Acute Kidney Injury & Critical Care Nephrology,You are meeting with your hospital’s chief inf...,Which ONE of the following individuals is MOST...,A. A 35-year-old woman with serum creatinine t...,B,temp nan,Answer B: A 76-year-old woman with a serum cr...,primary_eval
6,NEPHRO-EVAL-0007,2831,Acute Kidney Injury & Critical Care Nephrology,The chief quality ofﬁcer at your institution a...,"Based on the existing literature, implementing...",A. A reduction in nephrotoxin exposure and eve...,B,temp nan,"Answer B: A reduction in RRT events, inpatien...",primary_eval
7,NEPHRO-EVAL-0008,2832,Acute Kidney Injury & Critical Care Nephrology,"Several weeks after joining a new practice, yo...",Which ONE of the following outcomes has been d...,"A. Fewer cases of severe AKI, but no differenc...",C,temp nan,"Answer C: Fewer cases of severe AKI, a higher...",primary_eval
8,NEPHRO-EVAL-0009,2833,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2833:\nImplementing a Kidney Disease ...,A. Shorter length of mechanical ventilation\nB...,B,temp nan,Answer B: A decreased rate of stage 2 and 3 AKI,primary_eval
9,NEPHRO-EVAL-0010,2834,Acute Kidney Injury & Critical Care Nephrology,A 28-year-old woman with ESRD receives a de-ce...,This patient’s urine output in response to int...,A. Acute renal allograft rejection\nB. 1-year ...,C,temp nan,Answer C: Delayed graft function after renal ...,primary_eval



HELD-OUT EVALUATION
Shape: (160, 9)
Columns: ['BenchmarkID', 'ID', 'Subject', 'Context', 'Question', 'Choices', 'Answer', 'Solution', 'explanation']


,BenchmarkID,ID,Subject,Context,Question,Choices,Answer,Solution,explanation
0,NEPHRO-HOLDOUT-0001,2828,Acute Kidney Injury & Critical Care Nephrology,A 63-year-old kidney transplant recipient is s...,"In this setting, administration of bicarbonate...",A. Increased risk of mortality\nB. Increased r...,C,temp nan,Answer C: Decreased risk of progressive AKI r...
1,NEPHRO-HOLDOUT-0002,2848,Acute Kidney Injury & Critical Care Nephrology,A 65-year-old man with stage G3a CKD is seen i...,Which ONE of the following should you tell his...,A. Off-pump surgery in eligible patients is as...,A,temp nan,Answer A: Off-pump surgery in eligible patien...
2,NEPHRO-HOLDOUT-0003,5444,Acute Kidney Injury & Critical Care Nephrology,\nA 72-year-old man with hepatitis C–related c...,Which ONE of the following is the MOST likely ...,1. Abdominal compartment syndrome\n2. Acute tu...,D,temp nan,D. Hepatorenal syndrome
3,NEPHRO-HOLDOUT-0004,5448,Acute Kidney Injury & Critical Care Nephrology,\nYou are asked to see a 63-year-old man in co...,Which ONE of the following is MOST likely to p...,1. A fractional excretion of urea > 50%\n2. A ...,C,temp nan,C. Urine volume <200 ml over 2 hours after 1....
4,NEPHRO-HOLDOUT-0005,5455,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 5455\nIn which ONE of the following c...,1. Coronary artery bypass surgery\n2. Open abd...,C,temp nan,C. Radiocontrast exposure
5,NEPHRO-HOLDOUT-0006,5457,Acute Kidney Injury & Critical Care Nephrology,\nYou are asked to see a 68-year-old man with ...,Which ONE of the following is the MOST appropr...,1. Give a ﬂuid bolus and reassess in 1 hour\n2...,A,temp nan,A. Give a fluid bolus and reassess in 1 hour
6,NEPHRO-HOLDOUT-0007,5463,Acute Kidney Injury & Critical Care Nephrology,\nA 34-year-old combat victim is ﬂown to a mil...,Which ONE of the following places this trauma ...,1. Chest injury\n2. Head injury\n3. Abdominal ...,C,temp nan,C. Abdominal injury
7,NEPHRO-HOLDOUT-0008,5467,Acute Kidney Injury & Critical Care Nephrology,\nA 47-year-old woman with advanced heart fail...,Which ONE of the following would you advise re...,1. She is likely to experience at least a tran...,A,temp nan,A. She is likely to experience at least a tran...
8,NEPHRO-HOLDOUT-0009,5471,Acute Kidney Injury & Critical Care Nephrology,\nAn 80-year-old man is hospitalized for AKI. ...,Which ONE of the following diagnostic tests is...,1. Serum creatine kinase\n2. Urine eosinophils...,C,temp nan,C. Kidney ultrasound
9,NEPHRO-HOLDOUT-0010,5510,Acute Kidney Injury & Critical Care Nephrology,"\nA 75-year-old woman with diabetes, hypertens...",What strategies can you implement to prevent AKI?,A. Check urine sodium and risk-stratify...,C,temp nan,C. Give 0.9% sodium chloride before and after ...


In [72]:
# Compact manifest inspection

print("Manifest shape:", manifest_df.shape)
print("Manifest columns:", list(manifest_df.columns))

print("\nFirst 10 rows:")
display(manifest_df.head(10))

print("\nData types:")
display(manifest_df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(
    manifest_df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing")
)

Manifest shape: (796, 10)
Manifest columns: ['BenchmarkID', 'ID', 'Subject', 'Context', 'Question', 'Choices', 'Answer', 'Solution', 'explanation', 'split']

First 10 rows:


,BenchmarkID,ID,Subject,Context,Question,Choices,Answer,Solution,explanation,split
0,NEPHRO-EVAL-0001,2824,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2824:\nThe transfusion of longer-term...,1. Increased risk of transfusion-related react...,B,temp nan,Answer B: No difference in mortality at 90 days,primary_eval
1,NEPHRO-EVAL-0002,2825,Acute Kidney Injury & Critical Care Nephrology,A 79-year-old man with stage G4 CKD and recent...,"Based on data from the NUTRIREA-2 study, enter...",A. Decreased incidence of systemic bacteremia\...,B,temp nan,Answer B: Increased incidence of gastrointest...,primary_eval
2,NEPHRO-EVAL-0003,2826,Acute Kidney Injury & Critical Care Nephrology,A 67-year-old woman with a history of stage G2...,Which ONE of the following is CORRECT about th...,1. She is at decreased risk for inpatient mort...,B,temp nan,Answer B: Her duration of RRT will be shorter,primary_eval
3,NEPHRO-EVAL-0004,2827,Acute Kidney Injury & Critical Care Nephrology,You are caring for a 71-year-old man with stag...,Which ONE of the following is associated with ...,A. A decreased risk of surgical site infection...,B,temp nan,Answer B: An increased risk of AKI requiring RRT,primary_eval
4,NEPHRO-EVAL-0005,2829,Acute Kidney Injury & Critical Care Nephrology,You are admitting a 65-year-old kidney trans-p...,Which ONE of the following is the MOST importa...,A. Administration of a bolus of 30 ml/kg 0.9% ...,D,temp nan,Answer D: Early adminstration of appropriate ...,primary_eval
5,NEPHRO-EVAL-0006,2830,Acute Kidney Injury & Critical Care Nephrology,You are meeting with your hospital’s chief inf...,Which ONE of the following individuals is MOST...,A. A 35-year-old woman with serum creatinine t...,B,temp nan,Answer B: A 76-year-old woman with a serum cr...,primary_eval
6,NEPHRO-EVAL-0007,2831,Acute Kidney Injury & Critical Care Nephrology,The chief quality ofﬁcer at your institution a...,"Based on the existing literature, implementing...",A. A reduction in nephrotoxin exposure and eve...,B,temp nan,"Answer B: A reduction in RRT events, inpatien...",primary_eval
7,NEPHRO-EVAL-0008,2832,Acute Kidney Injury & Critical Care Nephrology,"Several weeks after joining a new practice, yo...",Which ONE of the following outcomes has been d...,"A. Fewer cases of severe AKI, but no differenc...",C,temp nan,"Answer C: Fewer cases of severe AKI, a higher...",primary_eval
8,NEPHRO-EVAL-0009,2833,Acute Kidney Injury & Critical Care Nephrology,NaN,Question 2833:\nImplementing a Kidney Disease ...,A. Shorter length of mechanical ventilation\nB...,B,temp nan,Answer B: A decreased rate of stage 2 and 3 AKI,primary_eval
9,NEPHRO-EVAL-0010,2834,Acute Kidney Injury & Critical Care Nephrology,A 28-year-old woman with ESRD receives a de-ce...,This patient’s urine output in response to int...,A. Acute renal allograft rejection\nB. 1-year ...,C,temp nan,Answer C: Delayed graft function after renal ...,primary_eval



Data types:


,dtype
BenchmarkID,object
ID,int64
Subject,object
Context,object
Question,object
Choices,object
Answer,object
Solution,object
explanation,object
split,object



Missing values:


,Missing
Context,48
BenchmarkID,0
ID,0
Subject,0
Question,0
Choices,0
Answer,0
Solution,0
explanation,0
split,0


In [73]:
# Inspect the official benchmark split assignments

print("Official split distribution")
print("===========================")

split_counts = (
    manifest_df["split"]
    .value_counts()
    .rename_axis("Split")
    .reset_index(name="Questions")
)

display(split_counts)

print("\nSubject distribution by split")
print("=============================")

split_subject_counts = (
    manifest_df
    .groupby(["split", "Subject"])
    .size()
    .reset_index(name="Questions")
)

display(split_subject_counts)

Official split distribution


,Split,Questions
0,primary_eval,636
1,held_out,160



Subject distribution by split


,split,Subject,Questions
0,held_out,Acute Kidney Injury & Critical Care Nephrology,18
1,held_out,Chronic Kidney Disease,6
2,held_out,Chronic Kidney Disease and Progression,6
3,held_out,Disorder of Divalent Ions Renal Bone Disease a...,6
4,held_out,Disorders of Divalent Ions Renal Bone Disease ...,5
5,held_out,"Disorders of Divalent Ions, Renal Bone Disease...",11
6,held_out,ESRD & Dialysis,6
7,held_out,Electrolyte Acid-Base Disorders,6
8,held_out,Electrolytes and Acid-Base Disorders,6
9,held_out,End Stage Renal Disease & Dialysis,6
